# SatQuery: train the high-resolution models on a free Kaggle GPU

Trains the two models that answer questions about sharp aerial photos, both on VRSBench:

* **hr_grounder**: finds the object a sentence describes (the app's current one scores 14.5% Acc@0.5)
* **hr_vqa**: answers questions about the photo (the app's current one scores 54.0%)

Both are small models served on the free CPU deployment; only training uses the GPU.

**Before you press Run all** (right-hand panel → Settings):
1. **Accelerator:** GPU T4 x1 (or P100)
2. **Internet:** On (needs a phone-verified Kaggle account)

Expected time: about 1-1.5 h (most of it is the 12 GB download and encoding the images once).
Output: `satquery_hr_models.zip` in the Output tab, with each model and its test scores. The last cell
says which models beat the ones in the app.

In [ ]:
# 1. Code (the public repo) and a GPU check
!nvidia-smi -L
!rm -rf /tmp/SatQuery && git clone -q --depth 1 https://github.com/Vansh-Vaid/SatQuery.git /tmp/SatQuery
!cd /tmp/SatQuery && git log --oneline -1

In [ ]:
# 2. VRSBench from Hugging Face (about 12 GB, kept in /tmp so it does not count against the 20 GB output limit)
import os, time
from huggingface_hub import hf_hub_download

ROOT = "/tmp/vrsbench"
os.makedirs(ROOT, exist_ok=True)
t0 = time.time()
for f in ["VRSBench_train.json", "VRSBench_EVAL_vqa.json", "VRSBench_EVAL_referring.json",
          "Images_val.zip", "Images_train.zip"]:
    for attempt in range(3):
        try:
            hf_hub_download("xiang709/VRSBench", f, repo_type="dataset", local_dir=ROOT)
            break
        except Exception as e:
            print(f"{f}: attempt {attempt + 1} failed ({e}); retrying")
            time.sleep(10)
    else:
        raise RuntimeError(f"could not download {f}; is Internet switched on in Settings?")
    print(f"{f}: {os.path.getsize(os.path.join(ROOT, f)) / 1e9:.2f} GB  ({(time.time() - t0) / 60:.1f} min)")

In [ ]:
# 3. Grounder: encodes every image once into 14x14 feature maps (reused by step 4), then trains
!cd /tmp/SatQuery && python training/train_vrsbench_grounding.py --root /tmp/vrsbench     --cache /tmp/vrsbench/features --encode-bs 64 --out /kaggle/working/vrsbench_ground.pt

In [ ]:
# 4. VQA model on the same feature maps (no images encoded again)
!cd /tmp/SatQuery && python training/train_vrsbench_vqa_dense.py --root /tmp/vrsbench     --cache /tmp/vrsbench/features --out /kaggle/working/vrsbench_vqa.pt

In [ ]:
# 5. Compare with the models in the app, and pack the results
import json, os, zipfile

W = "/kaggle/working"
checks = [("vrsbench_ground.pt", "test_acc_at_0.5", 0.1454, "grounder (Acc@0.5)"),
          ("vrsbench_vqa.pt", "test_accuracy", 0.5397, "VQA (accuracy)")]
with zipfile.ZipFile(f"{W}/satquery_hr_models.zip", "w") as z:
    for ck, key, base, name in checks:
        mp = f"{W}/{ck}.metrics.json"
        if not os.path.exists(mp):
            print(f"{name}: no result (see the error above)")
            continue
        new = json.load(open(mp))[key]
        verdict = "BETTER: use it" if new > base else "not better: keep the app's current model"
        print(f"{name}: new {new:.1%} vs app {base:.1%} -> {verdict}")
        z.write(f"{W}/{ck}", ck)
        z.write(mp, os.path.basename(mp))
print()
print("Download satquery_hr_models.zip from the Output tab.")

## Afterwards
Download `satquery_hr_models.zip` and unzip it into the project's `checkpoints/` folder (overwriting the
two files) **only for the models marked BETTER**. Then they go into the `models-v1` GitHub release and
the app is redeployed, so the live site uses them.